# Entrenamiento · clasificador de TEMA (Fase 4A)

- **Train:** `data/topic_synthetic.csv` (sintéticos, ~80 por categoría).
- **Test:** SOLO prompts reales etiquetados a mano (`data/topic_test_real.csv`) → se evalúa en `evaluate.ipynb`.
- Validación cruzada **5-fold agrupada** (`GroupKFold`): las variantes de un mismo prompt base nunca quedan en train y validación a la vez.
- Baseline: TF-IDF (palabras 1–2 + caracteres 2–5) + LogisticRegression.
- Principal: embeddings `paraphrase-multilingual-MiniLM-L12-v2` + LogisticRegression (`class_weight='balanced'`).

In [1]:
import sys, warnings, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, joblib, sklearn
import mlutils as U
U.set_seeds()
pd.set_option("display.max_colwidth", 120)


In [2]:
train = U.load_topic_train()
test_real = U.load_topic_test_real()
print(train["label"].value_counts().to_string())
print("grupos (prompts base):", train["group"].nunique())
print("test real etiquetado:", 0 if test_real is None else len(test_real))


label
consulta_general    80
personal            80
redaccion           80
programacion        80
academico           80
ciberseguridad      80
grupos (prompts base): 216
test real etiquetado: 0


## Baseline TF-IDF · elección de C con CV 5-fold agrupada (F1 macro)

In [3]:
from sklearn.model_selection import GroupKFold, cross_val_score, cross_val_predict
cv = GroupKFold(n_splits=5)
X_text, y, groups = train["text"].tolist(), train["label"].to_numpy(), train["group"].to_numpy()

def grid(make, X, Cs):
    rows = []
    for C in Cs:
        s = cross_val_score(make(C), X, y, groups=groups, cv=cv, scoring="f1_macro", n_jobs=1)
        rows.append({"C": C, "f1_macro_mean": s.mean(), "f1_macro_std": s.std()})
    return pd.DataFrame(rows)

g_tfidf = grid(lambda C: U.tfidf_pipeline(C, class_weight="balanced"), X_text, [0.3, 1, 3, 10, 30])
best_tfidf = g_tfidf.loc[g_tfidf.f1_macro_mean.idxmax()]
g_tfidf.round(4)


Out[3]: 
      C  f1_macro_mean  f1_macro_std
0   0.3         0.6076        0.0488
1   1.0         0.6123        0.0449
2   3.0         0.6102        0.0399
3  10.0         0.6079        0.0373
4  30.0         0.6068        0.0404


,C,f1_macro_mean,f1_macro_std
0,0.3,0.6076,0.0488
1,1.0,0.6123,0.0449
2,3.0,0.6102,0.0399
3,10.0,0.6079,0.0373
4,30.0,0.6068,0.0404


## Embeddings + LogisticRegression · elección de C

In [4]:
t0 = time.perf_counter()
X_emb = U.embed(X_text, cache="topic_train")
print(f"embeddings {X_emb.shape} en {time.perf_counter()-t0:.1f}s")
g_emb = grid(U.embedding_lr, X_emb, [0.1, 0.3, 1, 3, 10, 30])
best_emb = g_emb.loc[g_emb.f1_macro_mean.idxmax()]
g_emb.round(4)


embeddings (480, 384) en 10.0s
Out[4]: 
      C  f1_macro_mean  f1_macro_std
0   0.1         0.8170        0.0351
1   0.3         0.8087        0.0426
2   1.0         0.7992        0.0521
3   3.0         0.7900        0.0566
4  10.0         0.8002        0.0520
5  30.0         0.7911        0.0477


,C,f1_macro_mean,f1_macro_std
0,0.1,0.8170,0.0351
1,0.3,0.8087,0.0426
2,1.0,0.7992,0.0521
3,3.0,0.7900,0.0566
4,10.0,0.8002,0.0520
5,30.0,0.7911,0.0477


## Predicciones fuera de fold (para el análisis de errores mientras no exista el test real)

In [5]:
oof = train[["text", "label", "source"]].copy()
oof["pred_tfidf"] = cross_val_predict(U.tfidf_pipeline(best_tfidf.C, class_weight="balanced"), X_text, y, groups=groups, cv=cv)
oof["pred_emb"] = cross_val_predict(U.embedding_lr(best_emb.C), X_emb, y, groups=groups, cv=cv)
oof.to_csv(U.CACHE / "topic_oof.csv", index=False)
print("acierto OOF  tfidf:", (oof.pred_tfidf == oof.label).mean().round(3), " emb:", (oof.pred_emb == oof.label).mean().round(3))


acierto OOF  tfidf: 0.629  emb: 0.823


## Entrenamiento final y exportación a `ml/models/`

In [6]:
from datetime import datetime, timezone
meta = {"seed": U.SEED, "sklearn_version": sklearn.__version__, "trained_at": datetime.now(timezone.utc).isoformat(),
        "train_data": "ml/data/topic_synthetic.csv", "n_train": len(train)}

tfidf = U.tfidf_pipeline(best_tfidf.C, class_weight="balanced").fit(X_text, y)
joblib.dump({**meta, "kind": "tfidf+logreg", "pipeline": tfidf, "labels": list(tfidf.classes_),
             "C": float(best_tfidf.C)}, U.MODELS / "topic_tfidf.joblib")

lr = U.embedding_lr(best_emb.C).fit(X_emb, y)
joblib.dump({**meta, "kind": "embeddings+logreg", "embedding_model": U.EMB_MODEL, "normalize_embeddings": True,
             "clf": lr, "labels": list(lr.classes_), "C": float(best_emb.C)}, U.MODELS / "topic_clf.joblib")

U.update_metrics("topic_train", {
    "n_train": len(train), "cv": "GroupKFold(5) por prompt base",
    "tfidf": {"C": float(best_tfidf.C), "cv_f1_macro": round(float(best_tfidf.f1_macro_mean), 4), "cv_std": round(float(best_tfidf.f1_macro_std), 4)},
    "embeddings": {"C": float(best_emb.C), "cv_f1_macro": round(float(best_emb.f1_macro_mean), 4), "cv_std": round(float(best_emb.f1_macro_std), 4)},
})
sorted(p.name for p in U.MODELS.glob("topic_*.joblib"))


Out[6]: ['topic_clf.joblib', 'topic_tfidf.joblib']


['topic_clf.joblib', 'topic_tfidf.joblib']